In [ ]:
#PREPROCESSING THE DATASET TO CREATE TENSORS
#we are creating this to make sure our model will know 2 sec window ie 
#1 sec current window and 1 sec previous window
#ie  for index 1: [ features(index 0), features(index 1) ]
# also for index 2: [ features(index 1), features(index 2) ]

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Data.csv")
df = df.sort_values("timestamp").reset_index(drop=True)

In [3]:
#test the data type of dataset
print(df.dtypes)

timestamp          float64
switch_id            int64
src_ip                 str
dst_ip                 str
eth_type             int64
in_port              int64
out_port             int64
packet_count         int64
byte_count           int64
duration           float64
packet_rate        float64
byte_rate          float64
avg_packet_size    float64
label                int64
dtype: object


In [4]:
# FEATURE_COLUMNS = [
#     "packet_in_count",
#     "packet_in_rate",
#     "flow_mod_count",
#     "flow_removed_count",
#     "packet_in_to_flow_mod_ratio",
#     "flow_table_size",
#     "flow_table_utilization_ratio",
#     "flow_entry_growth_rate",
#     "unique_src_ip",
#     "unique_dst_ip",
#     "src_ip_entropy",
#     "dst_ip_entropy",
#     "tcp_syn_count",
#     "udp_packet_count",
#     "arp_packet_count",
#     "icmp_packet_count",
#     "avg_flow_duration",
#     "short_flow_ratio",
#     "packet_in_variance",
#     "burst_score"
# ]

#to undersatnd the metrics used above
# Control Plane Metrics
# packet_in_count
# packet_in_rate
# flow_mod_count
# flow_removed_count
# packet_in_to_flow_mod_ratio

# Flow Table Metrics
# flow_table_size
# flow_table_utilization_ratio
# flow_entry_growth_rate

# Distribution / Entropy Metrics
# unique_src_ip
# unique_dst_ip
# src_ip_entropy
# dst_ip_entropy

# Protocol Counters
# tcp_syn_count
# udp_packet_count
# arp_packet_count
# icmp_packet_count

# Flow Behavior
# avg_flow_duration
# short_flow_ratio

# Temporal / Burst
# packet_in_variance
# burst_score

# attack types we will do 
# Controller Flooding (Packet_In DDoS)
# Flow Table Exhaustion
# LOFT (Low-Rate DoS)
# ARP Spoof Burst
# Control-Plane Reflection Trigger



#for testing fn change after getting full data set with all these features
# Automatically select numeric columns except excluded ones
EXCLUDE_COLUMNS = ["timestamp", "switch_id", "label", "attack_id"]

FEATURE_COLUMNS = [
    col for col in df.columns
    if col not in EXCLUDE_COLUMNS
    and pd.api.types.is_numeric_dtype(df[col])
]


LABEL_COLUMN = "label"


print("Using features:", FEATURE_COLUMNS)


Using features: ['eth_type', 'in_port', 'out_port', 'packet_count', 'byte_count', 'duration', 'packet_rate', 'byte_rate', 'avg_packet_size']


In [5]:
grouped = df.groupby("switch_id")
print(df.columns.tolist())


['timestamp', 'switch_id', 'src_ip', 'dst_ip', 'eth_type', 'in_port', 'out_port', 'packet_count', 'byte_count', 'duration', 'packet_rate', 'byte_rate', 'avg_packet_size', 'label']


In [6]:
#window logic per switch 
WINDOW_SIZE = 2

X = []
y = []
window_switch_ids = []
window_timestamps = []

grouped = df.groupby("switch_id")

for switch_id, switch_df in grouped:
    
    switch_df = switch_df.sort_values("timestamp").reset_index(drop=True)
    
    features = switch_df[FEATURE_COLUMNS].values
    labels = switch_df["label"].values
    timestamps = switch_df["timestamp"].values
    
    for i in range(len(switch_df) - WINDOW_SIZE + 1):
        
        window = features[i:i+WINDOW_SIZE]
        label = labels[i+WINDOW_SIZE-1]
        ts = timestamps[i+WINDOW_SIZE-1]
        
        X.append(window)
        y.append(label)
        window_switch_ids.append(switch_id)
        window_timestamps.append(ts)

X = np.array(X)
y = np.array(y)
window_switch_ids = np.array(window_switch_ids)
window_timestamps = np.array(window_timestamps)

# Transpose for CNN
X = np.transpose(X, (0, 2, 1))


In [7]:
#Convert to Numpy
X = np.array(X)  # shape (batch, window, features)
y = np.array(y)

print("Shape before transpose:", X.shape)


Shape before transpose: (247830, 9, 2)


In [8]:
#CNN expects:(batch, features, window)
X = np.transpose(X, (0, 2, 1))
print("Final shape:", X.shape)


Final shape: (247830, 2, 9)


In [9]:
#save for training
np.save("X.npy", X.astype(np.float32))
np.save("y.npy", y.astype(np.int64))
np.save("window_switch_ids.npy", window_switch_ids)
np.save("window_timestamps.npy", window_timestamps)

print("Saved processed dataset dataset")

Saved processed dataset dataset


In [10]:
#to check balance of datsset normal vs attacks vs each attack
from collections import Counter
print(Counter(y))


Counter({np.int64(1): 172429, np.int64(0): 69260, np.int64(2): 6141})
